In [1]:
# Scientific computing.
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import IPython

# Resolve Utilities path from the actual notebook location.
# VSCode is fighting the entire time. Try/except for a failsafe. 
# Uses VS Code's injected variable to find the notebook file reliably.

# Ugh. https://github.com/microsoft/vscode-jupyter/issues/8475
try:
    notebook_path = Path(IPython.extract_module_locals()[1]["__vsc_ipynb_file__"])
    utilities_path = notebook_path.parent.parent / "Utilities"
except KeyError:
    # Fallback if run outside VS Code.
    utilities_path = Path().resolve() / "Notebooks" / "Utilities"

sys.path.append(str(utilities_path))

from SI_Utilities import (
    load_survey_data,
    load_tfidf_vectorizer,
    get_agreement_index,
    prepare_model_data_tfidf,
    Leader_R_Cols, 
    Primary_Pairs,
    build_summary_tfidf_df,
)

from Model_Utilities import (
    run_tfidf_nb, 
    split_and_train_tfidf_nb,
    run_tfidf_nb

)

# Machine learning.
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [2]:
# Anchor to the notebook's location to not hardcode paths.
Notebook_Dir = Path().resolve()
Project_Root = Notebook_Dir.parents[1]
Data_Dir = Project_Root / "Clean_Data_Resources"

# Load the parquet from Text_Parsing.ipynb.
Survey_df = pd.read_parquet(Data_Dir / "Survey_df_Text_Parsed.parquet")

# Load scale guide that maps each column/semester/discipline to its scale type.
Likert_Guide_df = pd.read_csv(Data_Dir / "Survey_Results_Likert_Guide.csv")

# Load column metadata.
with open(Data_Dir / "column_metadata.json", "r") as f:
    Column_Metadata = json.load(f)

# Compute averaged leader score.
Survey_df["R_Leader_Average_encoded"] = Survey_df[Leader_R_Cols].mean(axis=1)

In [3]:
# Confirm import and data loaded correctly.
# Print count of working agreement rows. 
test_idx = get_agreement_index(
    "R_Collaborative_Activities_encoded",
    Survey_df,
    Likert_Guide_df
)
print(f"Agreement rows for R_Collaborative_Activities: {len(test_idx)}")

Agreement rows for R_Collaborative_Activities: 31128


In [4]:
Agreement_df = Survey_df.loc[test_idx]

# Semester and year counts.
print(Agreement_df["Semester"].value_counts().to_string())
print(Agreement_df["Year"].value_counts().sort_index().to_string())

# Non-null response counts per T_ column within agreement rows.
Text_Cols = [
    "T_Collaboration_Understanding",
    "T_Leader_Performance_Suggestions",
    "T_Other_Suggestions",
    "T_Program_Overall_Suggestions",
]
for col in Text_Cols:
    count = Agreement_df[col].notna().sum()
    print(f"{col.replace('T_', '')}: {count}")

Semester
Fall      18293
Spring    12835
Year
2019    6921
2020    4346
2021    2432
2022    3881
2023    6603
2024    3267
2025    3678
Collaboration_Understanding: 26841
Leader_Performance_Suggestions: 24718
Other_Suggestions: 1423
Program_Overall_Suggestions: 21713


In [5]:
# Join lemmas back to strings for TfidfVectorizer input.
# TfidfVectorizer expects raw strings, not pre-tokenized lists.
Text_Cols = list(set(t_col for t_col, _ in Primary_Pairs))
for col in Text_Cols:
    Survey_df[col + "_lemma_str"] = Survey_df[col + "_lemmas"].apply(
        lambda x: " ".join(x) if len(x) > 0 else ""
    )

In [6]:
# Pre-load all vectorizers once to avoid repeated disk reads.
Text_Cols = list(set(t_col for t_col, _ in Primary_Pairs))
Vectorizers = {col: load_tfidf_vectorizer(col, Project_Root) for col in Text_Cols}

In [7]:
# Run Naive Bayes with TF-IDF features across all primary pairings.
TF_IDF_NB_Results = []

for t_col, r_col in Primary_Pairs:
    label = f"{t_col.replace('T_', '')} to {r_col.replace('_encoded', '').replace('R_', '')}."
    print(f"Running: {label}")
    result = run_tfidf_nb(
        t_col, r_col, label, Survey_df, Likert_Guide_df,
        Project_Root, vectorizer=Vectorizers[t_col]
    )
    if result:
        TF_IDF_NB_Results.append(result)
        print(f"Accuracy: {result['Accuracy']}. ROC_AUC: {result['ROC_AUC']}. MCC: {result['MCC']}")
    print()

Running: Collaboration_Understanding to Collaborative_Activities.
Accuracy: 0.968. ROC_AUC: 0.776. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Helps_Understanding.
Accuracy: 0.942. ROC_AUC: 0.814. MCC: 0.042

Running: Leader_Performance_Suggestions to Leader_Subject_Competence.
Accuracy: 0.976. ROC_AUC: 0.812. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Has_Plan.
Accuracy: 0.981. ROC_AUC: 0.766. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Willing_To_Help.
Accuracy: 0.978. ROC_AUC: 0.768. MCC: 0.0

Running: Leader_Performance_Suggestions to Leader_Average.
Accuracy: 0.973. ROC_AUC: 0.815. MCC: 0.0

Running: Other_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.886. ROC_AUC: 0.738. MCC: 0.142

Running: Program_Overall_Suggestions to Overall_Session_Helpfulness.
Accuracy: 0.936. ROC_AUC: 0.74. MCC: 0.0



In [8]:
# Flatten results into a summary table.
TF_IDF_NB_Summary_df = build_summary_tfidf_df(TF_IDF_NB_Results)
print(TF_IDF_NB_Summary_df.reset_index().to_string(index=False))

 index                                                       Pairing     N  Pos %  Neg %  Accuracy    F1  Precision  Recall  ROC AUC   MCC
     0      Collaboration_Understanding to Collaborative_Activities. 24990   96.8    3.2     0.968 0.984      0.968   1.000    0.776 0.000
     1 Leader_Performance_Suggestions to Leader_Helps_Understanding. 20371   94.2    5.8     0.942 0.970      0.942   1.000    0.814 0.042
     2  Leader_Performance_Suggestions to Leader_Subject_Competence. 21041   97.6    2.4     0.976 0.988      0.976   1.000    0.812 0.000
     3            Leader_Performance_Suggestions to Leader_Has_Plan. 21076   98.1    1.9     0.981 0.991      0.981   1.000    0.766 0.000
     4     Leader_Performance_Suggestions to Leader_Willing_To_Help. 20837   97.8    2.2     0.978 0.989      0.978   1.000    0.768 0.000
     5             Leader_Performance_Suggestions to Leader_Average. 21526   97.3    2.7     0.973 0.986      0.973   1.000    0.815 0.000
     6             Other_Su

In [11]:
# Test different thresholds to find where the model actually discriminates.
sample_responses = [
    "This SI session was incredibly helpful, I finally understand the material.",
    "The leader went too fast and I left more confused than when I arrived.",
    "It was okay I guess, nothing special.",
    "Absolutely terrible, the leader had no plan and wasted our time.",
    "Great review session, practice problems were perfect for the exam.",
]

thresholds = [0.5, 0.3, 0.2, 0.1, 0.05]

result = TF_IDF_NB_Results[0]
vectorizer = result["Vectorizer"]
nb_model = result["Model"]

for response in sample_responses:
    vec = vectorizer.transform([response])
    prob = nb_model.predict_proba(vec)[0][1]
    print(f"Response: {response[:60]}")
    for t in thresholds:
        prediction = "POSITIVE" if prob > t else "NEGATIVE"
        print(f"  threshold={t}: {prediction} | P(pos)={round(prob, 3)}")
    print()

Response: This SI session was incredibly helpful, I finally understand
  threshold=0.5: POSITIVE | P(pos)=0.98
  threshold=0.3: POSITIVE | P(pos)=0.98
  threshold=0.2: POSITIVE | P(pos)=0.98
  threshold=0.1: POSITIVE | P(pos)=0.98
  threshold=0.05: POSITIVE | P(pos)=0.98

Response: The leader went too fast and I left more confused than when 
  threshold=0.5: POSITIVE | P(pos)=0.982
  threshold=0.3: POSITIVE | P(pos)=0.982
  threshold=0.2: POSITIVE | P(pos)=0.982
  threshold=0.1: POSITIVE | P(pos)=0.982
  threshold=0.05: POSITIVE | P(pos)=0.982

Response: It was okay I guess, nothing special.
  threshold=0.5: POSITIVE | P(pos)=0.921
  threshold=0.3: POSITIVE | P(pos)=0.921
  threshold=0.2: POSITIVE | P(pos)=0.921
  threshold=0.1: POSITIVE | P(pos)=0.921
  threshold=0.05: POSITIVE | P(pos)=0.921

Response: Absolutely terrible, the leader had no plan and wasted our t
  threshold=0.5: POSITIVE | P(pos)=0.819
  threshold=0.3: POSITIVE | P(pos)=0.819
  threshold=0.2: POSITIVE | P(pos)=0.819
